# Chapter 72 — Generation From the Trained Character Model

Chapter 71 trained a tiny character-level GPT to predict the next character at every position.

This chapter turns those predictions into text one character at a time.

## Learning goals

By the end of this chapter, you will be able to:

- explain autoregressive generation;
- encode and validate a prompt;
- select final-position logits from the model output;
- convert logits to probabilities and sample one token;
- append a sampled token and repeat the process;
- keep generation deterministic with a dedicated random generator; and
- explain why only the newest context window remains visible to the model.

## Autoregressive generation

A **prompt** is the starting sequence given to a model.

**Autoregressive generation** samples one new token, appends it, and uses the enlarged sequence to predict the following token.

A **context window** is the portion of the sequence visible during one forward pass.

**Logits** are raw vocabulary scores, while **probabilities** are nonnegative values that sum to `1`.

At each step, generation performs this ordered process:

1. Keep at most the newest `context_length` token IDs.
2. Run those IDs through the model.
3. Select the vocabulary logits at the final position.
4. Apply softmax over the vocabulary dimension.
5. Sample one token ID and append it.

## Recreate the Chapter 71 model

Jupyter notebooks do not inherit live variables from one another when executed from clean kernels.

The next three cells therefore recreate the Chapter 71 data, architecture, and 1,000-update training run.

This prerequisite is intentionally compact because the new subject is generation.

In [1]:
import torch

device = "cpu"
model_config = {
    "context_length": 64,
    "embedding_dimension": 64,
    "number_of_attention_heads": 4,
    "number_of_transformer_blocks": 2,
    "dropout_rate": 0.1,
    "batch_size": 16,
    "learning_rate": 0.0003,
    "training_steps": 1000,
}

public_domain_text = """
Alice was beginning to get very tired of sitting by her sister on the bank,
and of having nothing to do. Once or twice she had peeped into the book her
sister was reading, but it had no pictures or conversations in it, and what is
the use of a book, thought Alice, without pictures or conversation?

So she was considering in her own mind, as well as she could, for the hot day
made her feel very sleepy and stupid, whether the pleasure of making a daisy
chain would be worth the trouble of getting up and picking the daisies, when
suddenly a white rabbit with pink eyes ran close by her.

There was nothing so very remarkable in that; nor did Alice think it so very
much out of the way to hear the rabbit say to itself, Oh dear! Oh dear! I shall
be too late! But when the rabbit actually took a watch out of its waistcoat
pocket, and looked at it, and then hurried on, Alice started to her feet.

The rabbit-hole went straight on like a tunnel for some way, and then dipped
suddenly down, so suddenly that Alice had not a moment to think about stopping
herself before she found herself falling down what seemed to be a very deep
well.

Either the well was very deep, or she fell very slowly, for she had plenty of
time as she went down to look about her and to wonder what was going to happen
next.
""".strip()

training_source_text = public_domain_text[: int(0.85 * len(public_domain_text))]
training_text = (training_source_text + "\n\n") * 20
characters = sorted(set(public_domain_text + "\n"))
character_to_id = {
    character: character_id for character_id, character in enumerate(characters)
}
id_to_character = {
    character_id: character for character, character_id in character_to_id.items()
}


def encode_text(text: str, token_to_id: dict[str, int]) -> list[int]:
    return [token_to_id[character] for character in text]


def decode_token_ids(token_ids: list[int], id_to_token: dict[int, str]) -> str:
    return "".join(id_to_token[token_id] for token_id in token_ids)


training_token_ids = torch.tensor(
    encode_text(training_text, character_to_id),
    dtype=torch.long,
    device=device,
)

print("device:", device)
print("vocabulary size:", len(characters))
print("training token shape:", training_token_ids.shape)

device: cpu
vocabulary size: 38
training token shape: torch.Size([22120])


The model below matches the combined-head causal transformer used in Chapter 71.

Its forward method returns logits shaped `[batch, context, vocabulary]` and an optional all-position loss.

In [2]:
import math


class MultiHeadCausalSelfAttention(torch.nn.Module):
    """Compute causal multi-head attention with combined projections."""

    embedding_dimension: int
    number_of_attention_heads: int
    head_size: int
    context_length: int
    query_projection: torch.nn.Linear
    key_projection: torch.nn.Linear
    value_projection: torch.nn.Linear
    attention_dropout: torch.nn.Dropout
    output_projection: torch.nn.Linear
    output_dropout: torch.nn.Dropout
    causal_mask: torch.Tensor

    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        if embedding_dimension % number_of_attention_heads != 0:
            raise ValueError(
                "embedding dimension must divide evenly by the head count."
            )

        self.embedding_dimension = embedding_dimension
        self.number_of_attention_heads = number_of_attention_heads
        self.head_size = embedding_dimension // number_of_attention_heads
        self.context_length = context_length
        self.query_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.key_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.value_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.attention_dropout = torch.nn.Dropout(dropout_rate)
        self.output_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension
        )
        self.output_dropout = torch.nn.Dropout(dropout_rate)
        self.register_buffer(
            "causal_mask",
            torch.tril(
                torch.ones(1, 1, context_length, context_length, dtype=torch.bool)
            ),
        )

    def _split_heads(self, vectors: torch.Tensor) -> torch.Tensor:
        batch_size, current_context_length, _ = vectors.shape
        return vectors.reshape(
            batch_size,
            current_context_length,
            self.number_of_attention_heads,
            self.head_size,
        ).transpose(1, 2)

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        batch_size, current_context_length, _ = input_values.shape
        queries = self._split_heads(self.query_projection(input_values))
        keys = self._split_heads(self.key_projection(input_values))
        values = self._split_heads(self.value_projection(input_values))
        scaled_scores = (queries @ keys.transpose(-2, -1)) / math.sqrt(self.head_size)
        allowed = self.causal_mask[
            :, :, :current_context_length, :current_context_length
        ]
        weights = torch.softmax(
            scaled_scores.masked_fill(~allowed, float("-inf")), dim=-1
        )
        attended_values = self.attention_dropout(weights) @ values
        concatenated = attended_values.transpose(1, 2).reshape(
            batch_size, current_context_length, self.embedding_dimension
        )
        return self.output_dropout(self.output_projection(concatenated))


class PositionWiseFeedForward(torch.nn.Module):
    """Apply a shared C-to-4C-to-C network at every position."""

    network: torch.nn.Sequential

    def __init__(self, embedding_dimension: int, dropout_rate: float) -> None:
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(embedding_dimension, 4 * embedding_dimension),
            torch.nn.ReLU(),
            torch.nn.Linear(4 * embedding_dimension, embedding_dimension),
            torch.nn.Dropout(dropout_rate),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        return self.network(input_values)


class TransformerBlock(torch.nn.Module):
    """Apply one pre-normalized causal transformer block."""

    first_layer_norm: torch.nn.LayerNorm
    attention: MultiHeadCausalSelfAttention
    second_layer_norm: torch.nn.LayerNorm
    feedforward: PositionWiseFeedForward

    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.first_layer_norm = torch.nn.LayerNorm(embedding_dimension)
        self.attention = MultiHeadCausalSelfAttention(
            embedding_dimension,
            number_of_attention_heads,
            context_length,
            dropout_rate,
        )
        self.second_layer_norm = torch.nn.LayerNorm(embedding_dimension)
        self.feedforward = PositionWiseFeedForward(embedding_dimension, dropout_rate)

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        attention_change = self.attention(self.first_layer_norm(input_values))
        values_after_attention = input_values + attention_change
        feedforward_change = self.feedforward(
            self.second_layer_norm(values_after_attention)
        )
        return values_after_attention + feedforward_change


class TinyGPT(torch.nn.Module):
    """Map character ID sequences to next-character logits and loss."""

    vocabulary_size: int
    context_length: int
    token_embedding_table: torch.nn.Embedding
    position_embedding_table: torch.nn.Embedding
    embedding_dropout: torch.nn.Dropout
    transformer_blocks: torch.nn.ModuleList
    final_layer_norm: torch.nn.LayerNorm
    output_layer: torch.nn.Linear

    def __init__(
        self,
        vocabulary_size: int,
        context_length: int,
        embedding_dimension: int,
        number_of_attention_heads: int,
        number_of_transformer_blocks: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.context_length = context_length
        self.token_embedding_table = torch.nn.Embedding(
            vocabulary_size, embedding_dimension
        )
        self.position_embedding_table = torch.nn.Embedding(
            context_length, embedding_dimension
        )
        self.embedding_dropout = torch.nn.Dropout(dropout_rate)
        self.transformer_blocks = torch.nn.ModuleList(
            [
                TransformerBlock(
                    embedding_dimension,
                    number_of_attention_heads,
                    context_length,
                    dropout_rate,
                )
                for _ in range(number_of_transformer_blocks)
            ]
        )
        self.final_layer_norm = torch.nn.LayerNorm(embedding_dimension)
        self.output_layer = torch.nn.Linear(embedding_dimension, vocabulary_size)

    def forward(
        self,
        input_token_ids: torch.Tensor,
        target_token_ids: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        if input_token_ids.ndim != 2:
            raise ValueError("input IDs must have shape [batch, context].")

        current_context_length = input_token_ids.shape[1]
        if current_context_length < 1:
            raise ValueError("input context cannot be empty.")
        if current_context_length > self.context_length:
            raise ValueError("input exceeds the model context length.")

        token_embeddings = self.token_embedding_table(input_token_ids)
        position_ids = torch.arange(
            current_context_length, device=input_token_ids.device
        )
        current_values = self.embedding_dropout(
            token_embeddings + self.position_embedding_table(position_ids)
        )
        for block in self.transformer_blocks:
            current_values = block(current_values)

        vocabulary_logits = self.output_layer(self.final_layer_norm(current_values))
        loss = None
        if target_token_ids is not None:
            if target_token_ids.shape != input_token_ids.shape:
                raise ValueError("targets must have the same shape as inputs.")
            loss = torch.nn.functional.cross_entropy(
                vocabulary_logits.reshape(-1, self.vocabulary_size),
                target_token_ids.reshape(-1),
            )
        return vocabulary_logits, loss

The deterministic training cell uses the same initialization, batches, and optimizer settings as Chapter 71.

The printed checkpoints confirm that this notebook is generating from learned parameters rather than a random model.

In [3]:
def get_gpt_training_batch(
    token_ids: torch.Tensor,
    batch_size: int,
    context_length: int,
    generator: torch.Generator,
) -> tuple[torch.Tensor, torch.Tensor]:
    number_of_valid_starts = token_ids.numel() - context_length
    start_indexes = torch.randint(
        0,
        number_of_valid_starts,
        (batch_size,),
        generator=generator,
        device=token_ids.device,
    )
    offsets = torch.arange(context_length, device=token_ids.device)
    input_indexes = start_indexes[:, None] + offsets[None, :]
    return token_ids[input_indexes], token_ids[input_indexes + 1]


torch.manual_seed(71)
model = TinyGPT(
    vocabulary_size=len(characters),
    context_length=model_config["context_length"],
    embedding_dimension=model_config["embedding_dimension"],
    number_of_attention_heads=model_config["number_of_attention_heads"],
    number_of_transformer_blocks=model_config["number_of_transformer_blocks"],
    dropout_rate=model_config["dropout_rate"],
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=model_config["learning_rate"])
batch_generator = torch.Generator(device=device).manual_seed(7103)
checkpoint_losses = {}
model.train()

for step in range(1, model_config["training_steps"] + 1):
    input_batch, target_batch = get_gpt_training_batch(
        training_token_ids,
        batch_size=model_config["batch_size"],
        context_length=model_config["context_length"],
        generator=batch_generator,
    )
    _, loss = model(input_batch, target_batch)
    if loss is None:
        raise RuntimeError("training requires targets.")

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step in {1, 250, 500, 750, 1000}:
        checkpoint_losses[step] = loss.item()
        print(f"step {step:4}: batch loss {loss.item():.4f}")

model.eval()
print("model training mode:", model.training)

step    1: batch loss 3.8101


step  250: batch loss 2.3531


step  500: batch loss 2.1988


step  750: batch loss 2.0458


step 1000: batch loss 1.9439
model training mode: False


## Encode a prompt

A model input includes a batch dimension even when generation starts from one prompt.

The prompt `Alice` therefore becomes a tensor shaped `[1, 5]`.

In [4]:
prompt = "Alice"
prompt_token_ids = encode_text(prompt, character_to_id)
generated_token_ids = torch.tensor([prompt_token_ids], dtype=torch.long, device=device)

print("prompt:", repr(prompt))
print("token IDs:", prompt_token_ids)
print("tensor shape:", generated_token_ids.shape)

prompt: 'Alice'
token IDs: [8, 25, 23, 17, 19]
tensor shape: torch.Size([1, 5])


## Inspect the next-token distribution

The model still returns logits for every input position because its architecture does not change between training and generation.

Only `vocabulary_logits[:, -1, :]` predicts the token immediately after the complete visible context.

Softmax converts those final-position logits into a categorical probability distribution over characters.

In [5]:
with torch.no_grad():
    vocabulary_logits, loss = model(generated_token_ids)

final_position_logits = vocabulary_logits[:, -1, :]
next_token_probabilities = torch.softmax(final_position_logits, dim=-1)
top_probabilities, top_token_ids = torch.topk(next_token_probabilities[0], k=8)

print("all logits shape:", vocabulary_logits.shape)
print("final-position logits shape:", final_position_logits.shape)
print("loss without targets:", loss)
print("probability sum:", next_token_probabilities.sum(dim=-1))
print()
print("rank | character | probability")
print("-" * 32)
for rank, (token_id, probability) in enumerate(
    zip(
        top_token_ids.tolist(),
        top_probabilities.tolist(),
        strict=True,
    ),
    start=1,
):
    print(f"{rank:>4} | {repr(id_to_character[token_id]):>9} | {probability:.6f}")

all logits shape: torch.Size([1, 5, 38])
final-position logits shape: torch.Size([1, 38])
loss without targets: None
probability sum: tensor([1.0000])

rank | character | probability
--------------------------------
   1 |       ' ' | 0.692684
   2 |       'r' | 0.079430
   3 |       'n' | 0.057887
   4 |       't' | 0.040069
   5 |       'e' | 0.030460
   6 |       's' | 0.017665
   7 |       ',' | 0.014170
   8 |       'd' | 0.013688


## Sample and append one character

`torch.multinomial` chooses one vocabulary ID according to the probability distribution.

A dedicated generator makes this sample reproducible without changing unrelated randomness.

In [6]:
one_step_generator = torch.Generator(device=device).manual_seed(72)
next_token_ids = torch.multinomial(
    next_token_probabilities,
    num_samples=1,
    generator=one_step_generator,
)
updated_token_ids = torch.cat([generated_token_ids, next_token_ids], dim=1)
sampled_token_id = next_token_ids.item()

print("sampled token ID:", sampled_token_id)
print("sampled character:", repr(id_to_character[sampled_token_id]))
print("updated shape:", updated_token_ids.shape)
print(
    "updated text:",
    repr(decode_token_ids(updated_token_ids[0].tolist(), id_to_character)),
)

sampled token ID: 3
sampled character: ','
updated shape: torch.Size([1, 6])
updated text: 'Alice,'


## Crop to the newest context window

The complete generated sequence is retained for the final decoded result.

The model input is separately cropped with `generated_token_ids[:, -model.context_length:]`.

A short sequence remains unchanged, while a longer sequence keeps only its newest token IDs.

In [7]:
toy_sequence = torch.arange(20).reshape(1, 20)
toy_context_length = 8
short_sequence = toy_sequence[:, :3]
cropped_short_sequence = short_sequence[:, -toy_context_length:]
cropped_long_sequence = toy_sequence[:, -toy_context_length:]

print("short sequence:", short_sequence)
print("cropped short sequence:", cropped_short_sequence)
print("long sequence:", toy_sequence)
print("cropped long sequence:", cropped_long_sequence)

short sequence: tensor([[0, 1, 2]])
cropped short sequence: tensor([[0, 1, 2]])
long sequence: tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19]])
cropped long sequence: tensor([[12, 13, 14, 15, 16, 17, 18, 19]])


## Build a reusable generator

The function below implements the full loop and optionally records selected steps for inspection.

It validates the prompt, disables gradients and dropout, uses a local random generator, and restores the model's previous mode afterward.

Each trace record stores lengths and the sampled character rather than copying every intermediate tensor.

In [8]:
from typing import NamedTuple


class GenerationRecord(NamedTuple):
    """Describe one selected autoregressive generation step."""

    step: int
    full_length_before_step: int
    context_length_used: int
    context_text: str
    sampled_token_id: int
    sampled_character: str


def validate_prompt(prompt: str, token_to_id: dict[str, int]) -> None:
    if not prompt:
        raise ValueError("prompt must contain at least one character.")

    unknown_characters = sorted(set(prompt) - set(token_to_id))
    if unknown_characters:
        raise ValueError(
            "prompt contains characters outside the vocabulary: "
            + repr(unknown_characters)
        )


@torch.no_grad()
def generate_text(
    model: TinyGPT,
    prompt: str,
    number_of_new_tokens: int,
    sampling_seed: int,
    steps_to_record: set[int] | None = None,
) -> tuple[str, list[GenerationRecord]]:
    if number_of_new_tokens < 0:
        raise ValueError("number_of_new_tokens must be nonnegative.")
    validate_prompt(prompt, character_to_id)

    selected_steps = steps_to_record if steps_to_record is not None else set()
    generated_ids = torch.tensor(
        [encode_text(prompt, character_to_id)],
        dtype=torch.long,
        device=device,
    )
    sampling_generator = torch.Generator(device=device).manual_seed(sampling_seed)
    records = []
    previous_training_mode = model.training
    model.eval()

    try:
        for step in range(1, number_of_new_tokens + 1):
            full_length_before_step = generated_ids.shape[1]
            context_ids = generated_ids[:, -model.context_length :]
            vocabulary_logits, _ = model(context_ids)
            final_position_logits = vocabulary_logits[:, -1, :]
            probabilities = torch.softmax(final_position_logits, dim=-1)
            next_token_ids = torch.multinomial(
                probabilities,
                num_samples=1,
                generator=sampling_generator,
            )
            generated_ids = torch.cat([generated_ids, next_token_ids], dim=1)

            if step in selected_steps:
                sampled_token_id = next_token_ids.item()
                records.append(
                    GenerationRecord(
                        step=step,
                        full_length_before_step=full_length_before_step,
                        context_length_used=context_ids.shape[1],
                        context_text=decode_token_ids(
                            context_ids[0].tolist(), id_to_character
                        ),
                        sampled_token_id=sampled_token_id,
                        sampled_character=id_to_character[sampled_token_id],
                    )
                )
    finally:
        model.train(previous_training_mode)

    generated_text = decode_token_ids(generated_ids[0].tolist(), id_to_character)
    return generated_text, records

## Inspect the context boundary

The prompt has five characters and the model context length is `64`.

Generation step `60` sees exactly `64` tokens, and step `61` is the first step that must discard an old token.

The selected trace makes that boundary explicit.

In [9]:
generated_text, generation_records = generate_text(
    model,
    prompt="Alice",
    number_of_new_tokens=100,
    sampling_seed=72,
    steps_to_record={1, 2, 59, 60, 61, 100},
)

print("step | full length before | context used | sampled")
print("-" * 58)
for record in generation_records:
    print(
        f"{record.step:>4} | "
        f"{record.full_length_before_step:>18} | "
        f"{record.context_length_used:>12} | "
        f"{repr(record.sampled_character)}"
    )

print()
print("step 60 context starts with:", repr(generation_records[3].context_text[:12]))
print("step 61 context starts with:", repr(generation_records[4].context_text[:12]))
print()
print("generated text:")
print(generated_text)

step | full length before | context used | sampled
----------------------------------------------------------
   1 |                  5 |            5 | ','
   2 |                  6 |            6 | '\n'
  59 |                 63 |           63 | 'o'
  60 |                 64 |           64 | 'f'
  61 |                 65 |           64 | 'e'
 100 |                104 |           64 | 'o'

step 60 context starts with: 'Alice,\nb to '
step 61 context starts with: 'lice,\nb to t'

generated text:
Alice,
b to ta inng ofer ondd thering a o waso., s
as d ren henofe otang d haloman ddean vldin sther Alfo


At step `61`, the full sequence has `65` tokens but the model still receives only `64`.

The complete generated result keeps growing even though the model can no longer condition on its oldest characters.

Because this model uses learned absolute position embeddings, the retained window is assigned positions `0` through `63` on every cropped forward pass.

## Start from a prompt longer than the context

A long prompt may be retained in the returned text, but only its final `context_length` characters influence the first sampled token.

In [10]:
long_prompt = public_domain_text[:90]
long_prompt_result, long_prompt_records = generate_text(
    model,
    prompt=long_prompt,
    number_of_new_tokens=1,
    sampling_seed=72,
    steps_to_record={1},
)
long_prompt_record = long_prompt_records[0]

print("full prompt length:", len(long_prompt))
print("context length used:", long_prompt_record.context_length_used)
print("visible prompt suffix matches:", end=" ")
print(long_prompt_record.context_text == long_prompt[-model.context_length :])
print("sampled character:", repr(long_prompt_record.sampled_character))
print("returned text length:", len(long_prompt_result))

full prompt length: 90
context length used: 64
visible prompt suffix matches: True
sampled character: 'a'
returned text length: 91


## Sampling and random seeds

Sampling is random but reproducible when the model, prompt, and seed are unchanged.

Different seeds usually select different paths through the model's probability distributions.

In [11]:
sample_seed_1, _ = generate_text(
    model, "Alice", number_of_new_tokens=80, sampling_seed=1
)
repeated_seed_1, _ = generate_text(
    model, "Alice", number_of_new_tokens=80, sampling_seed=1
)
sample_seed_2, _ = generate_text(
    model, "Alice", number_of_new_tokens=80, sampling_seed=2
)

print("same seed matches:", sample_seed_1 == repeated_seed_1)
print("different seeds differ:", sample_seed_1 != sample_seed_2)
print()
print("seed 1:")
print(sample_seed_1)
print()
print("seed 2:")
print(sample_seed_2)

same seed matches: True
different seeds differ: True

seed 1:
Alice bet serat atit ois, oof d d Oherelang bind veunlf, sen pelinde henondast On vel

seed 2:
Alice vesable cte t hit omate wanle teetore t-h hule of tublerain t or waderd d ang d


## Validate prompt characters

This simple encoder has no unknown-token ID, so every prompt character must occur in the training vocabulary.

The explicit validation error is clearer than a raw dictionary lookup failure.

In [12]:
for invalid_prompt in ["", "Alice 😊"]:
    try:
        generate_text(
            model,
            invalid_prompt,
            number_of_new_tokens=1,
            sampling_seed=72,
        )
    except ValueError as error:
        print(f"{invalid_prompt!r}: {error}")

'': prompt must contain at least one character.
'Alice 😊': prompt contains characters outside the vocabulary: ['😊']


## Training and generation differ

During training, real text supplies an input and a shifted target at every position.

Cross-entropy produces a loss, backpropagation produces gradients, and the optimizer changes parameters.

During generation, the model receives a prompt plus its own sampled tokens.

There are no targets, no loss, no gradients, and no parameter updates.

`model.eval()` disables dropout, while `torch.no_grad()` avoids building gradient graphs.

## Why the generated text remains messy

This model is small and was trained on a short repeated excerpt.

Character-level sampling also feeds every sampled mistake into later predictions.

The model can learn local spelling and punctuation patterns without learning reliable long-range meaning.

Generated quality is therefore a demonstration of the mechanism, not evidence of broad language ability.

## Common mistakes

- Do not pass more than `context_length` tokens to this model.
- Do not select logits from every position when choosing one next token.
- Do not apply softmax across the batch or context dimension.
- Do not forget to append the sampled token before the next step.
- Do not generate with dropout active.
- Do not build gradient graphs during generation.
- Do not assume an arbitrary prompt can be encoded by this vocabulary.
- Do not expect the oldest generated tokens to remain visible forever.

## Takeaways

Autoregressive generation repeatedly crops, predicts, samples, and appends.

The central tensor operations are:

```python
context_ids = generated_ids[:, -model.context_length:]
vocabulary_logits, _ = model(context_ids)
final_position_logits = vocabulary_logits[:, -1, :]
probabilities = torch.softmax(final_position_logits, dim=-1)
next_token_ids = torch.multinomial(probabilities, num_samples=1)
generated_ids = torch.cat([generated_ids, next_token_ids], dim=1)
```

> The generated sequence can grow indefinitely, but this model conditions only on its newest `context_length` tokens.

## What comes next

The next chapter will control how conservative or varied generation is.

It will compare greedy decoding, temperature, top-k sampling, repetition, and random seeds.